In [29]:
""" 
how ollama call external tool:

tool define: Basically a python function define.
tool schema preparation: then shcema preparation is needed, it is a json format, where tool specific parameters description is given, like, param type, param name etc
llm call: In the ollama, it can not call any tool directly, it will return again a json, where tool specific information will be present, like which tool should call, what are the parameter of this tool etc.
execution: after getting this json manual have to run the tool 
second llm call: where we can pass prompt + tool name + tool response, prepare the last output.

"""

' \nhow ollama call external tool:\n\ntool define: Basically a python function define.\ntool schema preparation: then shcema preparation is needed, it is a json format, where tool specific parameters description is given, like, param type, param name etc\nllm call: In the ollama, it can not call any tool directly, it will return again a json, where tool specific information will be present, like which tool should call, what are the parameter of this tool etc.\nexecution: after getting this json manual have to run the tool \nsecond llm call: where we can pass prompt + tool name + tool response, prepare the last output.\n\n'

In [30]:
import ollama

In [31]:
# prepare a dummy database
inventory_db = {
    "laptop":{"stock":5,"base_price":62000},
    "monitor":{"stock":0,"base_price":32000},
    "keyboard":{"stock":5,"base_price":1200}
}


In [32]:
#define tools
#tool 1 : checking database
def check_inventory(product_name):
    product_name = product_name.lower()
    if product_name in inventory_db:
        return inventory_db[product_name]
    return {"stock":0,"base_price":None}

# tool 2 : business logic for discount
def calculate_loyality_discount(base_price,years_as_customer):
    discount = min(years_as_customer*0.05,0.30)
    final_price = base_price*(1-discount)
    return round(final_price,2)


In [33]:
# Map the functions so that the script can call them by name

available_function = {
    "check_inventory":check_inventory,
    "calculate_loyality_discount":calculate_loyality_discount
}

In [34]:
# define tool schema

tools = [
    {
        "type":"function",
        "function":{
            "name":"check_inventory",
            "description":"Get stock and price for a product",
            "parameters":{
                "type":"object",
                "properties":{
                    "product_name":{"type":"string"}
                },
                "required":["product_name"]
            }
        }
    },
    {
        "type":"function",
        "function":{
            "name":"calculate_loyality_discount",
            "description":"calculate final price base on the loyalty years",
            "parameters":{
                "type":"object",
                "properties":{
                    "base_price":{"type":"integer"},
                    "years_as_customer":{"type":"integer"}
                }
            },
            "required":["base_price","years_as_customer"]
        }
    }
]

In [36]:
# query 1: I am a customer for 5 years. What will be the final price of a laptop?
# query 2: I want to buy an iphone, can you check the stock?

message= [
    {
      "role": "user",
      "content": "I want to buy a laptop, can you check the stock?"
    }
  ]
response = ollama.chat(
    model = "alexmoneo777/qwen3:8b",
    messages=message,
    tools=tools
)
print(response)

model='alexmoneo777/qwen3:8b' created_at='2026-03-15T17:29:18.082074277Z' done=True done_reason='stop' total_duration=18566456210 load_duration=12094374753 prompt_eval_count=209 prompt_eval_duration=4153924243 eval_count=19 eval_duration=2278826401 message=Message(role='assistant', content='', thinking=None, images=None, tool_name=None, tool_calls=[ToolCall(function=Function(name='check_inventory', arguments={'product_name': 'laptop'}))]) logprobs=None


In [37]:
response['message']

Message(role='assistant', content='', thinking=None, images=None, tool_name=None, tool_calls=[ToolCall(function=Function(name='check_inventory', arguments={'product_name': 'laptop'}))])

In [38]:
# check if the model asked to use any tool or not
tool_call = response['message'].get('tool_calls')
if tool_call:
    for tool in tool_call:
        tool_name = tool['function']['name']
        tool_agrs = tool['function']['arguments']

        function_to_call = available_function[tool_name]
        result = function_to_call(**tool_agrs)

        message.append(response['message'])
        message.append({
            'role':'tool',
            'content':str(result)
            })

In [39]:
message

[{'role': 'user',
  'content': 'I want to buy a laptop, can you check the stock?'},
 Message(role='assistant', content='', thinking=None, images=None, tool_name=None, tool_calls=[ToolCall(function=Function(name='check_inventory', arguments={'product_name': 'laptop'}))]),
 {'role': 'tool', 'content': "{'stock': 5, 'base_price': 62000}"}]

In [40]:
final_response = ollama.chat(
    model = "qwen3.5:9b",
    messages=message
)

In [41]:
print(final_response['message']['content'])

I checked the current stock for laptops, and here's what I found:

- **Available Stock:** 5 units
- **Base Price:** 62,000 (currency unit not specified, could be IDR or another currency)

These figures represent a general inventory snapshot. To give you the best recommendation, could you let me know:

1. **What brand or model** are you interested in? (e.g., MacBook, Dell XPS, HP Spectre, etc.)
2. **Any specific features** you're looking for? (e.g., RAM, storage, screen size, battery life)
3. **What's your intended use?** (e.g., gaming, office work, programming, graphic design)

This will help me tailor suggestions and ensure you get a laptop that fits your needs and budget!
